# Milestone 1, Task 1: Data Quality Report
**Team:** Pharma 1A (Break Through Tech AI Studio, Fall 2026) | **Host company:** WebMD | **Challenge Advisor:** Kimia Naeiji | **AI Studio Coach:** Anushka Naik
**Dataset:** `data/BreakThroughTech_training_data_1.xlsx` (417 rows, 28 columns)

**Project goal:** predict second-half (H2) campaign performance from January to June data. Per the Challenge Project Overview, the targets are:
- **Primary target: `total_spend`**, evaluated with MAPE (goal under 30%) and R² (goal above 0.55) on the held-out H2 set.
- **Secondary target: cost per engagement.** The overview calls it `meta_mapped_cpe`; the closest column in this file is `cpe`.

Every issue below is judged with those targets in mind, including which columns would leak the answer into a model.

This notebook documents every data quality issue we found, how many rows and columns each one affects, and the fix we propose. Sections follow our column assignments so each teammate can find their part quickly:

| Owner | Columns |
|---|---|
| Rhea | brand, sub_product_name, country, pillar |
| Emma | media_type, profession, specialty, campaign_start_date, campaign_end_date |
| Abhiram | unique_reach, unique_engaged_hcps, total_engagements, total_budget_spent, engagement_rate |
| Bakari | engagement_frequency, pre_NRx, post_NRx, pre_avg_NRx |
| Timothy | post_avg_NRx, pre_NRx_avg, npi_specialty, region, NRx_lift |
| Nida | NRx_lift_pct, cpe, total_spend, actual_spend_per_engagement, spend_per_NRx_lift |

Every issue is written to a running **issue log** that is summarized at the end and exported to `reports/data_quality_issues.csv`. The fixes themselves are applied in `02_data_cleaning.ipynb` (Task 2).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_PATH = "../data/BreakThroughTech_training_data_1.xlsx"
df = pd.read_excel(DATA_PATH)
N = len(df)
print(f"Rows: {N} | Columns: {df.shape[1]}")
df.head()

Rows: 417 | Columns: 28


,brand,sub_product_name,country,pillar,media_type,profession,specialty,campaign_start_date,campaign_end_date,unique_reach,unique_engaged_hcps,total_engagements,total_budget_spent,engagement_rate,engagement_frequency,pre_NRx,post_NRx,pre_avg_NRx,post_avg_NRx,pre_NRx_avg,npi_specialty,region,NRx_lift,NRx_lift_pct,cpe,total_spend,actual_spend_per_engagement,spend_per_NRx_lift
0,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,8,undefined,2024-07-31 10:48:34.000000 UTC,2099-12-31 23:59:59.999999 UTC,9270,1070,4449,173511.0,0.1154,4.16,946.88,1093.64,63.0600,72.8343,47.4714,Neurology,Southeast,146.77,15.5000,39.0,134355308.0,30198.99,915440.73
1,ABBVIE,BRAND ALERT,usa,search,EDITORIAL_VIDEOS,8,undefined,2024-08-19 17:16:44.000000 UTC,2099-12-31 23:59:59.999999 UTC,1959,380,1178,45942.0,0.1940,3.10,915.49,1027.54,48.2253,54.1279,62.6833,Cardiology,Southwest,112.05,12.2398,39.0,10667850.0,9055.90,95202.91
2,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,psychologist,urology,2023-11-16 15:54:10.000000 UTC,2099-12-31 23:59:59.999999 UTC,2860,403,1703,66417.0,0.1409,4.23,721.23,799.50,37.9804,42.1019,66.2213,Cardiology,Mid-Atlantic,78.27,10.8516,39.0,59103000.0,34705.23,755161.42
3,ABBVIE,BRAND ALERT,zaf,search,SPONSORED_VIDEOS,8,undefined,2024-11-22 10:27:31.000000 UTC,2099-12-31 23:59:59.999999 UTC,7390,910,1107,43173.0,0.1231,1.22,1022.39,1144.26,44.2658,49.5424,55.8293,Neurology,Northeast,121.87,11.9201,39.0,124487739.0,112455.05,1021479.17
4,ABBVIE,EMAIL ALERT,NaN,NaN,NaN,NaN,NaN,2023-11-16 17:38:18.000000 UTC,2099-12-31 23:59:59.999999 UTC,77570,26264,116027,348081.0,0.3386,4.42,630.86,852.37,34.7051,46.8907,39.3346,Endocrinology,Southwest,221.51,35.1116,3.0,892409925.0,7691.40,4028833.69


In [2]:
# Column assignments, in the order each person was given them.
ASSIGNMENTS = {
    "Rhea": ["brand", "sub_product_name", "country", "pillar"],
    "Emma": ["media_type", "profession", "specialty", "campaign_start_date", "campaign_end_date"],
    "Abhiram": ["unique_reach", "unique_engaged_hcps", "total_engagements", "total_budget_spent", "engagement_rate"],
    "Bakari": ["engagement_frequency", "pre_NRx", "post_NRx", "pre_avg_NRx"],
    "Timothy": ["post_avg_NRx", "pre_NRx_avg", "npi_specialty", "region", "NRx_lift"],
    "Nida": ["NRx_lift_pct", "cpe", "total_spend", "actual_spend_per_engagement", "spend_per_NRx_lift"],
}
COLUMN_ORDER = [c for cols in ASSIGNMENTS.values() for c in cols]
assert sorted(COLUMN_ORDER) == sorted(df.columns), "Every column should have exactly one owner"

# Running issue log. Every finding below is appended here.
# Issue IDs are assigned at the end, after sorting by assigned column order.
issues = []
def log_issue(owner, columns, issue, rows_affected, severity, proposed_fix):
    first_col = columns.split(",")[0].strip()
    assert first_col in ASSIGNMENTS[owner], f"{first_col} is not assigned to {owner}"
    issues.append({
        "column_position": COLUMN_ORDER.index(first_col) + 1,
        "owner": owner,
        "columns": columns,
        "issue": issue,
        "rows_affected": int(rows_affected),
        "pct_rows": round(100 * rows_affected / N, 1),
        "n_columns": len(columns.split(",")),
        "severity": severity,
        "proposed_fix": proposed_fix,
    })

## 1. Dataset overview

In [3]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "pct_null": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(),
})
overview

,dtype,non_null,nulls,pct_null,n_unique
brand,str,417,0,0.0,73
sub_product_name,str,417,0,0.0,11
country,str,361,56,13.4,8
pillar,str,361,56,13.4,4
media_type,str,337,80,19.2,2
profession,object,361,56,13.4,13
specialty,object,360,57,13.7,9
campaign_start_date,str,414,3,0.7,205
campaign_end_date,str,414,3,0.7,1
unique_reach,int64,417,0,0.0,410


In [4]:
print("Exact duplicate rows:", df.duplicated().sum())
cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
print("Duplicates on all non-numeric columns:", df.duplicated(cat_cols).sum())
ws = {c: (df[c].dropna().astype(str) != df[c].dropna().astype(str).str.strip()).sum() for c in cat_cols}
print("Leading or trailing whitespace by column:", {k: v for k, v in ws.items() if v} or "none")

Exact duplicate rows: 0
Duplicates on all non-numeric columns: 0
Leading or trailing whitespace by column: none


No exact duplicates and no stray whitespace, so those checks pass. The biggest structural finding is that **56 rows are missing country, pillar, media_type, profession and specialty all at once**, which tells us the gaps are a pattern in how the data was collected and not random.

In [5]:
null_pattern = df[["country", "pillar", "media_type", "profession", "specialty"]].isna()
print(null_pattern.value_counts().rename("rows").to_frame())
block = null_pattern.all(axis=1)
print("\nRows missing all five targeting fields:", block.sum())
print(df.loc[block, "sub_product_name"].value_counts())

                                                rows
country pillar media_type profession specialty      
False   False  False      False      False       336
True    True   True       True       True         56
False   False  True       False      False        24
               False      False      True          1

Rows missing all five targeting fields: 56
sub_product_name
EMAIL ALERT          46
TARGET LIST MEDIA     7
BRAND INTERCEPT       1
BRANDSPOT             1
DIRECT MAIL           1
Name: count, dtype: int64


In [6]:
log_issue("Rhea", "country, pillar",
          "Part of a 56-row block where country and pillar are missing together with Emma's targeting fields (mostly EMAIL ALERT). Missingness is structural, not random.",
          block.sum(), "High",
          "Fill with an explicit 'unknown' category and add a targeting_info_missing flag so models can learn from the pattern.")
log_issue("Emma", "media_type, profession, specialty",
          "Part of the same 56-row block: media_type, profession and specialty are missing on the same rows as country and pillar. Missingness is structural, not random.",
          block.sum(), "High",
          "Fill with an explicit 'unknown' category. Covered by the same targeting_info_missing flag.")

## 2. Rhea: brand, sub_product_name, country, pillar
### 2.1 brand

In [7]:
print("Distinct raw brand values:", df["brand"].nunique())
df["brand"].value_counts().to_frame("rows")

Distinct raw brand values: 73


,rows
brand,
MERCK,107
BMS,27
LILLY,21
BOEHRINGER INGELHEIM,17
GILEAD,14
...,...
STEMLINE,1
SUMITOMO PHARMA,1
SUMITOMO_GEMTESA,1


The `brand` column mixes three different kinds of information:

1. **Same company, different spelling:** `BMS` and `BRISTOL-MYERS SQUIBB`, `BOEHRINGER INGELHEIM` and `BOEHRINGER-INGELHEIM`, `NOVO NORDISK` and `NOVONORDISK_*`, `VIIV` and `VIIV HEALTHCARE`, `SERONO` and `EMD SERONO`, `RIGEL` and `RIGEL PHARMACEUTICALS`, `HORIZON` and `HORIZON THERAPEUTICS`, `DAY ONE BIOPHARMACEUTICALS` and `DAY_ONE_BIO`, `JAZZ PHARMACEUTICALS` and `JAZZ_*`.
2. **Company plus drug name glued together:** `LILLY_MOUNJARO_MEDSCAPE`, `ASTRAZENECA_ENHERTU`, `JANSSEN_DARZALEX`, `PFIZER_PAXLOVID`, and so on.
3. **Channel or vendor tags at the end:** `_MEDSCAPE`, `_MDALERT`, `_ZS`, `_BA`, `_EA`, plus indication tags like `TREMFYAPSA` (psoriatic arthritis) and `TREMFYAPSO` (psoriasis).

Left as is, a model would treat `BMS` and `BRISTOL-MYERS SQUIBB` as two unrelated advertisers and would spread the Lilly signal across three categories.

In [8]:
b = df["brand"]
underscore = b.str.contains("_")
spelling_groups = {
    "BRISTOL MYERS SQUIBB": ["BMS", "BRISTOL-MYERS SQUIBB"],
    "BOEHRINGER INGELHEIM": ["BOEHRINGER INGELHEIM", "BOEHRINGER-INGELHEIM"],
    "EMD SERONO": ["EMD SERONO", "SERONO"],
    "HORIZON": ["HORIZON", "HORIZON THERAPEUTICS"],
    "JANSSEN": ["JANSSEN", "JANSSEN BIOTECH", "JANSSEN CNS", "JANSSEN PHARMACEUTICALS", "J&J"],
    "NOVO NORDISK": ["NOVO NORDISK"],
    "RIGEL": ["RIGEL", "RIGEL PHARMACEUTICALS"],
    "VIIV HEALTHCARE": ["VIIV", "VIIV HEALTHCARE"],
    "DAY ONE BIOPHARMACEUTICALS": ["DAY ONE BIOPHARMACEUTICALS"],
    "JAZZ PHARMACEUTICALS": ["JAZZ PHARMACEUTICALS"],
    "OTSUKA": ["OTSUKA PHARMACEUTICAL"],
    "SUMITOMO PHARMA": ["SUMITOMO PHARMA"],
}
spelling_issue = b.isin([v for g, vals in spelling_groups.items() if len(vals) > 1 for v in vals])
brand_affected = (underscore | spelling_issue)
print("Rows with a compound (underscore) brand value:", underscore.sum())
print("Rows in a company that appears under more than one spelling:", spelling_issue.sum())
print("Total rows affected by brand inconsistency:", brand_affected.sum())

Rows with a compound (underscore) brand value: 87
Rows in a company that appears under more than one spelling: 96
Total rows affected by brand inconsistency: 183


In [9]:
log_issue("Rhea", "brand",
          f"Inconsistent naming: {df['brand'].nunique()} raw values mix spelling variants of the same company, company+drug compounds and channel tags (_MEDSCAPE, _MDALERT, _ZS, _BA, _EA).",
          brand_affected.sum(), "High",
          "Map every raw value to three clean columns: company (standardized name), product (drug name if present) and channel_tag. Keep brand_raw for auditing. Group companies with fewer than 5 rows as OTHER when encoding.")

### 2.2 sub_product_name

In [10]:
df["sub_product_name"].value_counts().to_frame("rows")

,rows
sub_product_name,
BRAND ALERT,151
EMAIL ALERT,145
TARGET LIST MEDIA,31
INFOSITE,28
BRANDSPOT,27
BRAND PLAY,11
POINT OF CONTACT,11
BRAND INTERCEPT,6
BRANDSTORY,5


In [11]:
rare_sub = df["sub_product_name"].map(df["sub_product_name"].value_counts()) < 5
print("Rows in sub_product categories with fewer than 5 rows:", rare_sub.sum())
log_issue("Rhea", "sub_product_name",
          "No nulls and consistent casing, but two categories are singletons (CONDITION MESSAGE, DIRECT MAIL), which cannot be learned reliably.",
          rare_sub.sum(), "Low",
          "Keep values as is. Group categories with fewer than 5 rows into 'OTHER' only in the encoded modeling file.")

Rows in sub_product categories with fewer than 5 rows: 2


### 2.3 country

In [12]:
print(df["country"].value_counts(dropna=False).to_frame("rows"))

         rows
country      
usa       201
zaf        95
NaN        56
vgb        32
vir        18
zwe         5
zmb         4
vnm         3
ind         3


In [13]:
cnt = df["country"].value_counts()
rare_country = df["country"].map(cnt) < 10
print("Null country:", df["country"].isna().sum())
print("Rows in countries with fewer than 10 rows:", rare_country.sum())
log_issue("Rhea", "country",
          f"{df['country'].isna().sum()} nulls ({df['country'].isna().mean():.1%}). Values are lowercase ISO-3 codes (usa, zaf, vgb, vir...) with no readable names.",
          df["country"].isna().sum(), "Medium",
          "Fill nulls with 'UNKNOWN', convert codes to uppercase ISO-3 and add a country_name column.")
log_issue("Rhea", "country",
          "Sparse categories: vnm and ind (3 rows each), zmb (4) and zwe (5) are too small to model on their own.",
          rare_country.sum(), "Low",
          "Group countries with fewer than 10 rows into 'OTHER' in the encoded file.")

Null country: 56
Rows in countries with fewer than 10 rows: 15


### 2.4 pillar

In [14]:
df["pillar"].value_counts(dropna=False).to_frame("rows")

,rows
pillar,
search,235
reference and tools,97
NaN,56
news and perspectives,27
other,2


In [15]:
log_issue("Rhea", "pillar",
          f"{df['pillar'].isna().sum()} nulls (same rows as the missing country block). 'other' has only 2 rows.",
          df["pillar"].isna().sum() + (df["pillar"] == "other").sum(), "Medium",
          "Fill nulls with 'unknown'. Keep 'other' as is and let it fall into the rare bucket during encoding.")

## 3. Emma: media_type, profession, specialty, campaign_start_date, campaign_end_date
### 3.1 media_type

In [16]:
print(df["media_type"].value_counts(dropna=False))
extra = df["media_type"].isna() & ~block
print("\nNull media_type outside the 56-row block:", extra.sum())
print(df.loc[extra, "brand"].value_counts())

media_type
EDITORIAL_VIDEOS    243
SPONSORED_VIDEOS     94
NaN                  80
Name: count, dtype: int64

Null media_type outside the 56-row block: 24
brand
LILLY       12
MERCK        6
NOVARTIS     6
Name: count, dtype: int64


In [17]:
log_issue("Emma", "media_type",
          f"{df['media_type'].isna().sum()} nulls: 56 from the missing block plus {extra.sum()} more on LILLY, MERCK and NOVARTIS rows that otherwise have full targeting data.",
          df["media_type"].isna().sum(), "Medium",
          "Fill with 'unknown' and lowercase values (editorial_videos, sponsored_videos).")

### 3.2 profession

In [18]:
prof = df["profession"]
print(prof.map(lambda v: type(v).__name__).value_counts())
prof.map(repr).value_counts().to_frame("rows")

profession
int      296
str       65
float     56
Name: count, dtype: int64


,rows
profession,
8,162
99,107
nan,56
'psychologist',45
'physician assistant',16
19,6
20,5
11,4
'physician',4


In [19]:
is_code = prof.map(lambda v: isinstance(v, (int, float, np.integer)) and not pd.isna(v))
print("Numeric profession codes:", is_code.sum(), "| Text labels:", (prof.notna() & ~is_code).sum(), "| Null:", prof.isna().sum())
log_issue("Emma", "profession",
          "Mixed data types: numeric codes (8, 99, 10-23) sit in the same column as text labels (psychologist, physician assistant, physician). Codes have no lookup table.",
          is_code.sum(), "High",
          "Treat every value as a category label (code_8, code_99, psychologist...). Codes are nominal, not ordinal, so they must never be used as numbers. Ask WebMD for the code dictionary.")
log_issue("Emma", "profession",
          f"{prof.isna().sum()} nulls (all inside the 56-row block).",
          prof.isna().sum(), "Medium", "Fill with 'unknown'.")

Numeric profession codes: 296 | Text labels: 65 | Null: 56


### 3.3 specialty

In [20]:
spec = df["specialty"]
print(spec.map(repr).value_counts())
print()
print(pd.crosstab(prof.astype(str), spec.astype(str)))

specialty
'undefined'            295
nan                     57
'urology'               46
'vascular medicine'      8
77                       3
35                       2
45                       2
59                       2
70                       1
32                       1
Name: count, dtype: int64

specialty            32  35  45  59  70  77  undefined  urology  vascular medicine
profession                                                                        
10                    1   2   0   0   0   0          0        0                  0
11                    0   0   2   0   0   2          0        0                  0
12                    0   0   0   2   0   0          1        0                  0
15                    0   0   0   0   0   0          3        0                  0
19                    0   0   0   0   0   0          6        0                  0
20                    0   0   0   0   0   0          5        0                  0
23                    0   0  

In [21]:
undefined = spec.astype(str).str.lower().eq("undefined")
spec_missing = undefined | spec.isna()
print(f"'undefined' placeholder: {undefined.sum()} | null: {spec.isna().sum()} | effectively missing: {spec_missing.sum()} ({spec_missing.mean():.1%})")
odd = (prof == "psychologist") & (spec == "urology")
print("psychologist + urology rows:", odd.sum())
log_issue("Emma", "specialty",
          f"'undefined' is a hidden null. Together with real nulls, {spec_missing.mean():.0%} of rows carry no specialty information.",
          spec_missing.sum(), "High",
          "Convert 'undefined' and nulls to 'unknown' and add a has_specialty flag. Use npi_specialty (100% filled) as the main specialty feature.")
log_issue("Emma", "specialty",
          "Mixed types again: numeric codes (32, 35, 45, 59, 70, 77) alongside text.",
          spec.map(lambda v: isinstance(v, (int, float, np.integer)) and not pd.isna(v)).sum(), "Medium",
          "Store as string labels (code_77 etc.), never as numbers.")
log_issue("Emma", "profession, specialty",
          "Implausible pairings: psychologist paired with urology specialty.",
          odd.sum(), "Low",
          "Keep the rows but note the pairing to WebMD. Do not build features that depend on profession x specialty combinations.")

'undefined' placeholder: 295 | null: 57 | effectively missing: 352 (84.4%)
psychologist + urology rows: 36


### 3.4 campaign_start_date and campaign_end_date

In [22]:
print(df[["campaign_start_date", "campaign_end_date"]].head(3))
start = pd.to_datetime(df["campaign_start_date"].str.replace(" UTC", "", regex=False), errors="coerce")
print("\nStart date stored as:", df["campaign_start_date"].dtype, "| unparseable or null:", start.isna().sum())
print("Start range:", start.min(), "to", start.max())
print(df["campaign_end_date"].value_counts(dropna=False))

              campaign_start_date               campaign_end_date
0  2024-07-31 10:48:34.000000 UTC  2099-12-31 23:59:59.999999 UTC
1  2024-08-19 17:16:44.000000 UTC  2099-12-31 23:59:59.999999 UTC
2  2023-11-16 15:54:10.000000 UTC  2099-12-31 23:59:59.999999 UTC

Start date stored as: str | unparseable or null: 3
Start range: 2020-12-17 23:00:40 to 2026-04-17 14:58:22
campaign_end_date
2099-12-31 23:59:59.999999 UTC    414
NaN                                 3
Name: count, dtype: int64


In [23]:
sentinel = df["campaign_end_date"].astype(str).str.startswith("2099")
log_issue("Emma", "campaign_start_date",
          "Dates are stored as text with a ' UTC' suffix instead of a datetime type. 3 rows are null.",
          df["campaign_start_date"].isna().sum(), "Medium",
          "Parse to datetime. Impute the 3 nulls with the median start date for the same company and add a start_date_imputed flag. Derive start_year, start_quarter, start_month and campaign_age_days.")
log_issue("Emma", "campaign_end_date",
          f"{sentinel.sum()} rows ({sentinel.mean():.1%}) use the placeholder 2099-12-31 23:59:59, and the other 3 are null. There is no real end date anywhere, so duration cannot be computed.",
          sentinel.sum() + df['campaign_end_date'].isna().sum(), "High",
          "Drop the column. It carries no information (it is effectively constant). Use start-date features instead.")

## 4. Abhiram: unique_reach, unique_engaged_hcps, total_engagements, total_budget_spent, engagement_rate

In [24]:
cols = ["unique_reach", "unique_engaged_hcps", "total_engagements", "total_budget_spent", "engagement_rate"]
def outlier_table(cols):
    rows = []
    for c in cols:
        s = df[c].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        rows.append({"column": c, "nulls": df[c].isna().sum(), "zeros": (df[c] == 0).sum(),
                     "min": s.min(), "median": s.median(), "max": s.max(), "skew": round(s.skew(), 2),
                     "iqr_outliers": ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()})
    return pd.DataFrame(rows).set_index("column")
outlier_table(cols)

,nulls,zeros,min,median,max,skew,iqr_outliers
column,,,,,,,
unique_reach,0,0,514.000,7258.0000,291110.000,3.27,31
unique_engaged_hcps,0,0,49.000,1616.0000,141813.000,4.23,55
total_engagements,0,0,135.000,4390.0000,395199.000,4.02,54
total_budget_spent,0,2,0.000,77883.0000,9364485.000,5.77,49
engagement_rate,0,0,0.056,0.2397,0.549,0.52,0


**Note on `total_engagements`:** our milestone plan expected this column to be 100% null. In this version of the file it is **fully populated** (0 nulls) and it reconciles exactly with `engagement_frequency x unique_engaged_hcps`. So no drop is needed. We should confirm with our coach that everyone is working from the same file version.

In [25]:
chk = pd.DataFrame({
    "engaged <= reach": (df.unique_engaged_hcps <= df.unique_reach).all(),
    "engagement_rate = engaged / reach": ((df.unique_engaged_hcps / df.unique_reach - df.engagement_rate).abs() <= 0.001).all(),
    "engagement_frequency = engagements / engaged": ((df.total_engagements / df.unique_engaged_hcps - df.engagement_frequency).abs() <= 0.01).all(),
}, index=["passes"]).T
chk

,passes
engaged <= reach,True
engagement_rate = engaged / reach,True
engagement_frequency = engagements / engaged,True


In [26]:
zero_budget = df["total_budget_spent"] == 0
budget_mismatch = (df["total_engagements"] * df["cpe"] - df["total_budget_spent"]).abs() > 1
print("Zero budget rows:", zero_budget.sum())
print("Rows where budget != cpe x engagements (cpe known):", budget_mismatch.sum())
df.loc[budget_mismatch, ["brand", "sub_product_name", "cpe", "total_engagements", "total_budget_spent"]]

Zero budget rows: 2
Rows where budget != cpe x engagements (cpe known): 7


,brand,sub_product_name,cpe,total_engagements,total_budget_spent
109,GILEAD,BRAND ALERT,39.0,209,39.94
114,GILEAD,EMAIL ALERT,3.0,757,24.26
143,JANSSEN BIOTECH,BRAND ALERT,39.0,184,0.77
146,JANSSEN BIOTECH,EMAIL ALERT,3.0,643,0.00
173,LILLY,CONDITION MESSAGE,11.0,595,0.00
227,MERCK,BRAND ALERT,39.0,1398,299.59
236,MERCK,BRAND ALERT,39.0,332,98.49


In [27]:
skewed = ["unique_reach", "unique_engaged_hcps", "total_engagements", "total_budget_spent"]
ot = outlier_table(skewed)
log_issue("Abhiram", ", ".join(skewed),
          f"Strong right skew (skew 3.3 to 5.8) with {int(ot['iqr_outliers'].max())} IQR outliers in the worst column. A handful of very large campaigns dominate.",
          int(((df[skewed] > df[skewed].quantile(.75) + 1.5*(df[skewed].quantile(.75)-df[skewed].quantile(.25)))).any(axis=1).sum()), "Medium",
          "Keep the rows (they are real campaigns, not errors). Add log1p versions of each column for modeling.")
log_issue("Abhiram", "total_budget_spent",
          "Budget does not match cpe x engagements: 2 rows have $0 budget and 5 more have near-zero budgets (under $300) for hundreds of engagements.",
          (budget_mismatch | zero_budget).sum(), "Medium",
          "Keep the rows but add a budget_anomaly flag so they can be excluded from cost-efficiency analysis.")
log_issue("Abhiram", "total_budget_spent, total_engagements",
          "Leakage for the CPE target: cpe = total_budget_spent / total_engagements, so a model given both columns can calculate CPE directly. For the total_spend target they are fair inputs (budget is planned before the campaign and is the strongest single predictor of spend, log correlation 0.84).",
          N, "High",
          "Use both as features for total_spend. When predicting CPE, leave out total_budget_spent (and its log) so the model cannot rebuild the answer.")
log_issue("Abhiram", "total_engagements",
          "Milestone plan expected this column to be 100% null, but in this file it has 0 nulls and passes all consistency checks.",
          0, "Info",
          "No drop needed. Confirm the file version with our coach so the team is aligned.")
log_issue("Abhiram", "engagement_rate",
          "Checked against unique_engaged_hcps / unique_reach: 0 mismatches. No issue.",
          0, "Info", "No action.")

## 5. Bakari: engagement_frequency, pre_NRx, post_NRx, pre_avg_NRx

In [28]:
outlier_table(["engagement_frequency", "pre_NRx", "post_NRx", "pre_avg_NRx"])

,nulls,zeros,min,median,max,skew,iqr_outliers
column,,,,,,,
engagement_frequency,0,0,1.2000,2.9600,4.5000,-0.14,0
pre_NRx,0,0,35.0800,972.6200,3975.7300,0.96,0
post_NRx,0,0,41.9100,1167.5800,5138.2200,0.97,3
pre_avg_NRx,0,0,2.2251,60.9465,487.4121,1.85,22


In [29]:
periods_pre = df.pre_NRx / df.pre_avg_NRx
periods_post = df.post_NRx / df.post_avg_NRx
print("pre_NRx / pre_avg_NRx (implied number of periods):")
print(periods_pre.describe().round(2))
print("\nMax gap between pre and post implied periods:", (periods_pre - periods_post).abs().max().round(4))

pre_NRx / pre_avg_NRx (implied number of periods):
count    417.00
mean      16.73
std        4.92
min        8.04
25%       12.35
50%       17.22
75%       20.83
max       25.00
dtype: float64

Max gap between pre and post implied periods: 0.003


`pre_avg_NRx` is `pre_NRx` divided by a campaign-specific number of periods (between 8 and 25), and the same divisor is used for `post_avg_NRx`. That is a clean, consistent relationship. `engagement_frequency` has no nulls, no outliers and matches its formula.

**Planning-time note:** `post_NRx` is measured after the campaign has run. Our model is meant to forecast H2 performance before H2 begins, so post-campaign values would not be available at that point and should not be model inputs.

In [30]:
log_issue("Bakari", "engagement_frequency, pre_NRx, pre_avg_NRx",
          "No nulls. engagement_frequency matches its formula. pre_avg_NRx = pre_NRx / n_periods with a consistent divisor. Moderate skew only.",
          0, "Info", "No cleaning needed. Optionally add n_periods = pre_NRx / pre_avg_NRx as a feature.")
log_issue("Bakari", "post_NRx",
          "Measured after the campaign, so it is not known when H2 budgets are being planned. Using it as an input would give the model information it will not have in practice.",
          N, "Medium",
          "Keep in cleaned_df for EDA and the prescription-lift story, but exclude from the feature sets for total_spend and CPE.")

## 6. Timothy: post_avg_NRx, pre_NRx_avg, npi_specialty, region, NRx_lift

In [31]:
print(df[["pre_NRx", "pre_avg_NRx", "pre_NRx_avg", "post_avg_NRx"]].corr().round(3))
print("\npre_NRx / pre_NRx_avg:")
print((df.pre_NRx / df.pre_NRx_avg).describe().round(2))

              pre_NRx  pre_avg_NRx  pre_NRx_avg  post_avg_NRx
pre_NRx         1.000        0.870        0.890         0.871
pre_avg_NRx     0.870        1.000        0.780         0.995
pre_NRx_avg     0.890        0.780        1.000         0.779
post_avg_NRx    0.871        0.995        0.779         1.000

pre_NRx / pre_NRx_avg:
count    417.00
mean      20.16
std        5.75
min       10.01
25%       15.08
50%       20.25
75%       25.27
max       29.99
dtype: float64


In [32]:
log_issue("Timothy", "pre_NRx_avg",
          "Two columns with almost the same name hold different values (correlation 0.78). pre_avg_NRx follows a clear formula; pre_NRx_avg does not, and its definition is undocumented.",
          N, "Medium",
          "Keep both for now, flag for WebMD to define pre_NRx_avg. Avoid putting both in the same model until the meaning is confirmed.")
log_issue("Timothy", "post_avg_NRx",
          "Post-campaign value (post_NRx / n_periods). Not known at planning time, same as post_NRx.",
          N, "Medium", "Keep for EDA, exclude from the feature sets for total_spend and CPE.")

In [33]:
print(df["npi_specialty"].value_counts())

npi_specialty
Nephrology                  48
Cardiology                  40
Primary Care                36
Internal Medicine           35
Rheumatology                33
Endocrinology               31
Oncology                    31
Dermatology                 30
Hematology                  30
Gastroenterology            28
Pulmonology                 27
Neurology                   24
Urology                     23
3320 - Anesthesiologists     1
Name: count, dtype: int64


In [34]:
bad_npi = df["npi_specialty"].str.contains(r"\d", regex=True)
log_issue("Timothy", "npi_specialty",
          "One value uses a different format with a numeric code: '3320 - Anesthesiologists'. All other values are plain specialty names.",
          bad_npi.sum(), "Low", "Rename to 'Anesthesiology' to match the rest of the column.")

In [35]:
print(df["region"].value_counts())
print()
print(pd.crosstab(df["country"].fillna("NULL"), df["region"]))

region
Mountain              64
South Atlantic        51
Mid-Atlantic          50
Northeast             50
Southwest             46
Pacific               41
West                  40
Southeast             39
Midwest               35
West South Central     1
Name: count, dtype: int64

region   Mid-Atlantic  Midwest  Mountain  Northeast  Pacific  South Atlantic  Southeast  Southwest  West  West South Central
country                                                                                                                     
NULL                6        4         5          6        9              11          4          5     5                   1
ind                 0        3         0          0        0               0          0          0     0                   0
usa                30       15        32         26       20              18         20         20    20                   0
vgb                 5        1         7          1        4               8          2    

In [36]:
non_us = df["country"].notna() & (df["country"] != "usa")
wsc = df["region"] == "West South Central"
log_issue("Timothy", "region",
          "Mixed taxonomies: US Census divisions (South Atlantic, Mid-Atlantic, Mountain, West South Central) sit next to informal regions (Southeast, Southwest, West, Midwest) that overlap them. 'West South Central' is a singleton.",
          wsc.sum(), "Low",
          "Map 'West South Central' to 'Southwest' (its largest state is Texas). Keep the other labels, and note the overlap for WebMD.")
log_issue("Timothy", "region",
          "US regions are assigned to non-US countries (South Africa, India, Vietnam, Zambia, Zimbabwe, Virgin Islands). Region likely describes the HCP audience, not the country, but this is unconfirmed.",
          non_us.sum(), "Medium",
          "Keep both columns and add a region_country_mismatch flag. Ask WebMD what region represents.")

In [37]:
lift_diff = (df.post_NRx - df.pre_NRx - df.NRx_lift).abs()
print("NRx_lift vs (post_NRx - pre_NRx): rows off by more than 0.005:", (lift_diff > 0.005).sum(), "| largest gap:", lift_diff.max().round(3))
log_issue("Timothy", "NRx_lift",
          "Rounding differences of +/-0.01 against post_NRx - pre_NRx. No real error.",
          (lift_diff > 0.005).sum(), "Info", "No action. Use the provided NRx_lift column as is for EDA. It is post-campaign, so it stays out of the feature sets.")

NRx_lift vs (post_NRx - pre_NRx): rows off by more than 0.005: 106 | largest gap: 0.01


## 7. Nida: NRx_lift_pct, cpe, total_spend, actual_spend_per_engagement, spend_per_NRx_lift
This section holds both project targets (`total_spend` and `cpe`), so the checks here go a little deeper.

In [38]:
pct_diff = ((df.post_NRx - df.pre_NRx) / df.pre_NRx * 100 - df.NRx_lift_pct).abs()
pct_diff2 = (df.NRx_lift / df.pre_NRx * 100 - df.NRx_lift_pct).abs()
print("NRx_lift_pct vs (post-pre)/pre: off by > 0.01 pts:", (pct_diff > 0.01).sum())
print("NRx_lift_pct vs NRx_lift/pre:    off by > 0.01 pts:", (pct_diff2 > 0.01).sum())

NRx_lift_pct vs (post-pre)/pre: off by > 0.01 pts: 4
NRx_lift_pct vs NRx_lift/pre:    off by > 0.01 pts: 0


In [39]:
log_issue("Nida", "NRx_lift_pct",
          "Matches NRx_lift / pre_NRx exactly. 4 rows differ slightly from (post-pre)/pre because of the rounding in NRx_lift.",
          (pct_diff > 0.01).sum(), "Info", "No action. Post-campaign, so keep for EDA and the business story but out of the feature sets.")

In [40]:
print(pd.crosstab(df["sub_product_name"], df["cpe"].fillna(-1)).rename(columns={-1: "NULL"}))
implied = (df.total_budget_spent / df.total_engagements).round(2)
print("\nImplied cpe (budget / engagements) where cpe is null:")
print(implied[df.cpe.isna()].value_counts())

cpe                NULL  3.0  11.0  39.0  68.0
sub_product_name                              
BRAND ALERT           0    0     0   151     0
BRAND INTERCEPT       6    0     0     0     0
BRAND PLAY           11    0     0     0     0
BRANDSPOT             0    0     0     0    27
BRANDSTORY            5    0     0     0     0
CONDITION MESSAGE     0    0     1     0     0
DIRECT MAIL           1    0     0     0     0
EMAIL ALERT           0  145     0     0     0
INFOSITE             28    0     0     0     0
POINT OF CONTACT     11    0     0     0     0
TARGET LIST MEDIA    31    0     0     0     0

Implied cpe (budget / engagements) where cpe is null:
39.00    92
0.02      1
Name: count, dtype: int64


**CPE as a target:** wherever `cpe` is filled in, it is one fixed rate per product (every BRAND ALERT row is 39, every EMAIL ALERT row is 3). That means `sub_product_name` alone predicts known CPE perfectly, which would make CPE a trivial target. The overview names `meta_mapped_cpe`, which may be a different, campaign-level measure, so we should confirm with our Challenge Advisor which field is meant.

In [41]:
print("Distinct cpe values per product (known rows only):")
print(df.groupby("sub_product_name")["cpe"].nunique())

Distinct cpe values per product (known rows only):
sub_product_name
BRAND ALERT          1
BRAND INTERCEPT      0
BRAND PLAY           0
BRANDSPOT            1
BRANDSTORY           0
CONDITION MESSAGE    1
DIRECT MAIL          0
EMAIL ALERT          1
INFOSITE             0
POINT OF CONTACT     0
TARGET LIST MEDIA    0
Name: cpe, dtype: int64


In [42]:
log_issue("Nida", "cpe",
          f"{df['cpe'].isna().sum()} nulls ({df['cpe'].isna().mean():.1%}). cpe is fixed per product (BRAND ALERT 39, EMAIL ALERT 3, BRANDSPOT 68) and missing for every other product. Budget / engagements shows the true value is 39 for 92 of the 93 rows.",
          df["cpe"].isna().sum(), "High",
          "Impute cpe = total_budget_spent / total_engagements (rounded to 2 decimals) and add a cpe_imputed flag.")
log_issue("Nida", "cpe",
          "As a target, known cpe is a fixed rate per product, so sub_product_name predicts it perfectly. The overview asks for meta_mapped_cpe, which is not in this file.",
          df["cpe"].notna().sum(), "High",
          "Confirm with the Challenge Advisor whether cpe is the intended CPE target or whether meta_mapped_cpe will be provided. Until then, treat total_spend as the main target.")

In [43]:
ratio = df.total_spend / df.total_budget_spent
print("total_spend / total_budget_spent:")
print(ratio.describe().round(1))
recon = (df.actual_spend_per_engagement * df.total_engagements / df.total_spend - 1).abs()
print("\ntotal_spend = actual_spend_per_engagement x total_engagements holds for", (recon < 0.001).sum(), "of", df.total_spend.notna().sum(), "rows")
missing_spend = df.loc[df.total_spend.isna(), ["brand", "sub_product_name", "total_engagements", "actual_spend_per_engagement"]].copy()
missing_spend["rebuilt_total_spend"] = (missing_spend.actual_spend_per_engagement * missing_spend.total_engagements).round(0)
print("\nMedian of total_spend across all other rows:", round(df.total_spend.median()))
missing_spend

total_spend / total_budget_spent:
count      412.0
mean      5130.7
std       8449.5
min          1.1
25%        642.4
50%       1823.0
75%       5786.7
max      72719.9
dtype: float64

total_spend = actual_spend_per_engagement x total_engagements holds for 406 of 414 rows

Median of total_spend across all other rows: 173784814


,brand,sub_product_name,total_engagements,actual_spend_per_engagement,rebuilt_total_spend
170,LILLY,BRANDSPOT,14621,11885.97,173784767.0
183,LILLY,TARGET LIST MEDIA,5885,29530.13,173784815.0
282,MERCK,INFOSITE,3285,52902.53,173784811.0


The 3 missing `total_spend` values can be rebuilt as `actual_spend_per_engagement x total_engagements`, but all three come out at about **173.78 million, which is exactly the median of `total_spend`**. That tells us the source system filled these rows with the median before calculating the per-engagement columns. The real spend for these campaigns is unknown, and since `total_spend` is our primary target, these rows cannot be trusted as training or test labels.

In [44]:
print("Smallest total_spend values:", df.total_spend.sort_values().head(10).round(0).tolist())
near_zero = df.total_spend < 1000
print("Rows with total_spend under 1,000:", near_zero.sum())
print("Skew of total_spend:", round(df.total_spend.skew(), 2), "| skew of log1p(total_spend):", round(np.log1p(df.total_spend).skew(), 2))

Smallest total_spend values: [0.0, 0.0, 1.0, 9.0, 37.0, 45.0, 150.0, 362.0, 532245.0, 750000.0]
Rows with total_spend under 1,000: 8
Skew of total_spend: 6.01 | skew of log1p(total_spend): -2.47


**Why this matters for MAPE:** MAPE divides each error by the true value. The 2 campaigns with a total_spend of 0 make MAPE undefined, and the 6 others under 1,000 (values like 1, 9 and 37) would turn a small absolute miss into an error of thousands of percent, swamping the average. The next value up is in the hundreds of thousands, so 1,000 is a clean cutoff.

In [45]:
log_issue("Nida", "total_spend",
          "3 nulls in the primary target (the same 3 rows missing campaign dates). Rebuilding them from actual_spend_per_engagement x total_engagements gives exactly the column median, so the source had already median-filled them. Their true spend is unknown.",
          df["total_spend"].isna().sum(), "High",
          "Fill with the rebuilt value so the file has no blanks, add a total_spend_imputed flag, and exclude these 3 rows from training and evaluation when total_spend is the target.")
log_issue("Nida", "total_spend",
          "8 rows have total_spend under 1,000, including 2 at exactly 0. MAPE is undefined at 0 and explodes near 0, which would distort the headline metric.",
          int((df.total_spend < 1000).sum()), "High",
          "Add a spend_near_zero flag. Exclude these rows when calculating MAPE (report how many were excluded) and model log1p(total_spend) to handle the wide range.")
log_issue("Nida", "total_spend",
          "Extreme range (0 to about 39 billion, skew 6.0). total_spend is also a median of about 1,800x total_budget_spent, so the two columns measure different things. The overview confirms spend runs from near zero to billions, but the difference from budget is not documented.",
          df["total_spend"].notna().sum(), "Medium",
          "Model the log of total_spend and convert predictions back for MAPE. Ask WebMD how total_spend differs from total_budget_spent.")

In [46]:
spl = (df.spend_per_NRx_lift / (df.total_spend / df.NRx_lift) - 1).abs()
print("spend_per_NRx_lift off by > 1% from total_spend / NRx_lift:", (spl > 0.01).sum())
print(df.loc[spl > 0.01, ["total_spend", "NRx_lift", "spend_per_NRx_lift"]])
print("Zeros:", {c: int((df[c] == 0).sum()) for c in ["total_spend", "actual_spend_per_engagement", "spend_per_NRx_lift"]})

spend_per_NRx_lift off by > 1% from total_spend / NRx_lift: 2
     total_spend  NRx_lift  spend_per_NRx_lift
119          9.0    104.89                0.09
143          1.0     66.93                0.01
Zeros: {'total_spend': 2, 'actual_spend_per_engagement': 2, 'spend_per_NRx_lift': 2}


In [47]:
log_issue("Nida", "actual_spend_per_engagement, spend_per_NRx_lift",
          "Leakage for the primary target: actual_spend_per_engagement = total_spend / total_engagements and spend_per_NRx_lift = total_spend / NRx_lift. Either one lets a model rebuild total_spend almost exactly. (Separately, 2 rows are off from the formula only through rounding, and 2 are zero.)",
          N, "High",
          "Keep in cleaned_df for EDA, but never use these columns (or their logs) as features for total_spend or CPE.")

## 8. Issue log summary
This is the full list of findings with rows affected and proposed fixes, listed in the order our columns were assigned: Rhea, Emma, Abhiram, Bakari, Timothy, then Nida, and within each person in their column order. It is also saved to `reports/data_quality_issues.csv` so the team can reference it in the cleaning decision log (Task 3).

In [48]:
# Sort by the order columns were assigned (Rhea, Emma, Abhiram, Bakari, Timothy, Nida),
# then number the issues in that order.
issue_log = (pd.DataFrame(issues)
             .sort_values("column_position", kind="stable")
             .reset_index(drop=True))
issue_log.insert(0, "issue_id", [f"DQ-{i+1:02d}" for i in range(len(issue_log))])
issue_log = issue_log.drop(columns="column_position")
issue_log.to_csv("../reports/data_quality_issues.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    display(issue_log)

,issue_id,owner,columns,issue,rows_affected,pct_rows,n_columns,severity,proposed_fix
0,DQ-01,Rhea,brand,"Inconsistent naming: 73 raw values mix spelling variants of the same company, company+drug compounds and channel tags (_MEDSCAPE, _MDALERT, _ZS, _BA, _EA).",183,43.9,1,High,"Map every raw value to three clean columns: company (standardized name), product (drug name if present) and channel_tag. Keep brand_raw for auditing. Group companies with fewer than 5 rows as OTHE..."
1,DQ-02,Rhea,sub_product_name,"No nulls and consistent casing, but two categories are singletons (CONDITION MESSAGE, DIRECT MAIL), which cannot be learned reliably.",2,0.5,1,Low,Keep values as is. Group categories with fewer than 5 rows into 'OTHER' only in the encoded modeling file.
2,DQ-03,Rhea,"country, pillar","Part of a 56-row block where country and pillar are missing together with Emma's targeting fields (mostly EMAIL ALERT). Missingness is structural, not random.",56,13.4,2,High,Fill with an explicit 'unknown' category and add a targeting_info_missing flag so models can learn from the pattern.
3,DQ-04,Rhea,country,"56 nulls (13.4%). Values are lowercase ISO-3 codes (usa, zaf, vgb, vir...) with no readable names.",56,13.4,1,Medium,"Fill nulls with 'UNKNOWN', convert codes to uppercase ISO-3 and add a country_name column."
4,DQ-05,Rhea,country,"Sparse categories: vnm and ind (3 rows each), zmb (4) and zwe (5) are too small to model on their own.",15,3.6,1,Low,Group countries with fewer than 10 rows into 'OTHER' in the encoded file.
5,DQ-06,Rhea,pillar,56 nulls (same rows as the missing country block). 'other' has only 2 rows.,58,13.9,1,Medium,Fill nulls with 'unknown'. Keep 'other' as is and let it fall into the rare bucket during encoding.
6,DQ-07,Emma,"media_type, profession, specialty","Part of the same 56-row block: media_type, profession and specialty are missing on the same rows as country and pillar. Missingness is structural, not random.",56,13.4,3,High,Fill with an explicit 'unknown' category. Covered by the same targeting_info_missing flag.
7,DQ-08,Emma,media_type,"80 nulls: 56 from the missing block plus 24 more on LILLY, MERCK and NOVARTIS rows that otherwise have full targeting data.",80,19.2,1,Medium,"Fill with 'unknown' and lowercase values (editorial_videos, sponsored_videos)."
8,DQ-09,Emma,profession,"Mixed data types: numeric codes (8, 99, 10-23) sit in the same column as text labels (psychologist, physician assistant, physician). Codes have no lookup table.",296,71.0,1,High,"Treat every value as a category label (code_8, code_99, psychologist...). Codes are nominal, not ordinal, so they must never be used as numbers. Ask WebMD for the code dictionary."
9,DQ-10,Emma,profession,56 nulls (all inside the 56-row block).,56,13.4,1,Medium,Fill with 'unknown'.


In [49]:
sev_order = ["High", "Medium", "Low", "Info"]
print(issue_log["severity"].value_counts().reindex(sev_order).fillna(0).astype(int).to_frame("issues"))
print()
print(issue_log.groupby("owner", sort=False).size().to_frame("issues"))

          issues
severity        
High          12
Medium        13
Low            5
Info           5

         issues
owner          
Rhea          6
Emma          9
Abhiram       5
Bakari        2
Timothy       6
Nida          7


## 9. Key takeaways
1. **Brand names need a full remap.** 73 raw values collapse into 34 companies once spelling variants, drug names and channel tags are separated.
2. **Missing targeting data is structural.** 56 rows are missing five fields at once, and `specialty` is effectively 84% empty once the `undefined` placeholder is counted. We will use explicit `unknown` categories and flags instead of dropping rows, because we only have 417 rows.
3. **Campaign end date is unusable.** Every value is the 2099 placeholder or null, so we drop it and build time features from the start date.
4. **Our primary target, `total_spend`, needs care.** 3 values were median-filled at the source (unknown true spend), and 8 values are under 1,000, which breaks MAPE. We flag both groups, model the log of spend, and leave those rows out of MAPE.
5. **CPE may be a trivial target in this file.** Known `cpe` is one fixed rate per product. We need to confirm whether `meta_mapped_cpe` is a different field.
6. **Watch for leakage.** For `total_spend`: `actual_spend_per_engagement`, `spend_per_NRx_lift` and their logs rebuild the target. For CPE: `total_budget_spent` together with `total_engagements` rebuilds it. Post-campaign columns (`post_NRx`, `post_avg_NRx`, `NRx_lift`, `NRx_lift_pct`) are not known at planning time. All of these stay in the cleaned file for EDA but out of the feature sets.
7. **`cpe` nulls can be recovered exactly** from budget / engagements.
8. **`total_engagements` is not null in this file**, which differs from our plan. No drop is needed.
9. **There is no H1/H2 marker in this file.** The overview describes training on January to June and testing on H2, but the only date is `campaign_start_date` (2020 to 2026). We need to confirm how the H2 held-out set will be provided.

**Questions for our Challenge Advisor:** whether `cpe` is the intended CPE target or `meta_mapped_cpe` will be provided; how the H2 test set will be delivered; how `total_spend` differs from `total_budget_spent`; code dictionaries for `profession` and `specialty`; the definition of `pre_NRx_avg` vs `pre_avg_NRx`; and what `region` represents for non-US rows.